In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("PairRDDs")
  .master("local[*]")
  .getOrCreate()

val sc = spark.sparkContext

// RDD de ventas como cadenas: "producto,región,importe"
val ventas = sc.parallelize(List(
  "Laptop,Norte,1200",
  "Teclado,Sur,45",
  "Monitor,Norte,350",
  "Laptop,Sur,1200",
  "Ratón,Norte,25",
  "Monitor,Sur,350",
  "Teclado,Norte,45"
))

// Convertimos a Pair RDD: (región, importe)
val ventasPorRegion = ventas.map { linea =>
  val campos = linea.split(",")
  val region  = campos(1)
  val importe = campos(2).toDouble
  (region, importe)   // ← tupla (clave, valor)
}

ventasPorRegion.collect().foreach(println)
// (Norte,1200.0)
// (Sur,45.0)
// (Norte,350.0)
// (Sur,1200.0)
// (Norte,25.0)
// (Sur,350.0)
// (Norte,45.0)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/27 11:03:30 INFO SparkContext: Running Spark version 4.1.1
26/04/27 11:03:30 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/27 11:03:30 INFO SparkContext: Java version 17.0.12+8-LTS-286
26/04/27 11:03:30 INFO ResourceUtils: ==============================================================
26/04/27 11:03:30 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/27 11:03:30 INFO ResourceUtils: ==============================================================
26/04/27 11:03:30 INFO SparkContext: Submitted application: PairRDDs
26/04/27 11:03:30 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/27 11:03:30 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/27 11:03:30 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/27 11:03:30 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/27 11:03:30 INFO SecurityManager: Changing view acls to: 

2026-04-27T09:03:30.699524200Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@38bd76b1
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@43911b78
ventas: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[0] at parallelize at cmd1.sc:12
ventasPorRegion: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[1] at map at cmd1.sc:23

In [2]:
val regiones = ventasPorRegion.keys
// RDD: "Norte", "Sur", "Norte", "Sur", ...

val importes = ventasPorRegion.values
// RDD: 1200.0, 45.0, 350.0, ...

println("Regiones distintas:")
regiones.distinct().collect().foreach(println)
// Norte
// Sur

Regiones distintas:
Sur
Norte


regiones: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[2] at keys at cmd2.sc:1
importes: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[3] at values at cmd2.sc:4

In [3]:
// Aplicar IVA (21%) solo al valor, sin tocar la clave
val ventasConIVA = ventasPorRegion.mapValues(importe => importe * 1.21)

ventasConIVA.collect().foreach(println)
// (Norte,1452.0)
// (Sur,54.45)
// (Norte,423.5)
// (Sur,1452.0)
// (Norte,30.25)
// (Sur,423.5)
// (Norte,54.45)

(Norte,1452.0)
(Sur,54.449999999999996)
(Norte,423.5)
(Sur,1452.0)
(Norte,30.25)
(Sur,423.5)
(Norte,54.449999999999996)


ventasConIVA: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[7] at mapValues at cmd3.sc:2

In [4]:
// Datos: (cliente, "producto1|producto2|producto3")
val pedidos = sc.parallelize(List(
  ("Ana",  "Laptop|Ratón"),
  ("Luis", "Teclado|Monitor|Altavoces"),
  ("Marta","Laptop")
))

// Expande cada pedido en filas individuales por producto
val productosPorCliente = pedidos.flatMapValues(productos => productos.split("\\|"))

productosPorCliente.collect().foreach(println)
// (Ana,Laptop)
// (Ana,Ratón)
// (Luis,Teclado)
// (Luis,Monitor)
// (Luis,Altavoces)
// (Marta,Laptop)

(Ana,Laptop)
(Ana,Ratón)
(Luis,Teclado)
(Luis,Monitor)
(Luis,Altavoces)
(Marta,Laptop)


pedidos: org.apache.spark.rdd.RDD[(String, String)] = ParallelCollectionRDD[8] at parallelize at cmd4.sc:2
productosPorCliente: org.apache.spark.rdd.RDD[(String, String)] = MapPartitionsRDD[9] at flatMapValues at cmd4.sc:9

In [5]:
// Total de ventas por región
val totalPorRegion = ventasPorRegion.reduceByKey(_ + _)

totalPorRegion.collect().foreach(println)
// (Norte,1620.0)
// (Sur,1595.0)

(Sur,1595.0)
(Norte,1620.0)


totalPorRegion: org.apache.spark.rdd.RDD[(String, Double)] = ShuffledRDD[10] at reduceByKey at cmd5.sc:2

In [6]:
// Queremos: (región → (suma de importes, número de ventas))
// Acumulador: (Double, Int) — tipo distinto al valor original Double

val resumenPorRegion = ventasPorRegion.aggregateByKey(
  (0.0, 0)                                    // valor inicial: (suma=0.0, count=0)
)(
  (acc, valor) => (acc._1 + valor, acc._2 + 1),   // combinar dentro de partición
  (acc1, acc2) => (acc1._1 + acc2._1, acc1._2 + acc2._2) // combinar entre particiones
)

resumenPorRegion.collect().foreach { case (region, (suma, count)) =>
  val media = suma / count
  println(f"$region: total=$suma%.2f, ventas=$count, media=$media%.2f")
}
// Norte: total=1620.00, ventas=4, media=405.00
// Sur:   total=1595.00, ventas=3, media=531.67

Sur: total=1595,00, ventas=3, media=531,67
Norte: total=1620,00, ventas=4, media=405,00


resumenPorRegion: org.apache.spark.rdd.RDD[(String, (Double, Int))] = ShuffledRDD[11] at aggregateByKey at cmd6.sc:6

In [7]:
// Objetivo: para cada región → List con todos los importes
val importesPorRegion = ventasPorRegion.combineByKey(
  (v: Double) => List(v),                       // createCombiner: primer valor → List
  (acc: List[Double], v: Double) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2 // mergeCombiners: unir Lists
)

importesPorRegion.collect().foreach { case (region, lista) =>
  println(s"$region → $lista")
}
// Norte → List(1200.0, 350.0, 25.0, 45.0)
// Sur   → List(45.0, 1200.0, 350.0)

Sur → List(45.0, 1200.0, 350.0)
Norte → List(1200.0, 350.0, 25.0, 45.0)


importesPorRegion: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[12] at combineByKey at cmd7.sc:5

In [8]:
val gruposPorRegion = ventasPorRegion.groupByKey()

gruposPorRegion.collect().foreach { case (region, valores) =>
  println(s"$region → ${valores.toList}")
}
// Norte → List(1200.0, 350.0, 25.0, 45.0)
// Sur   → List(45.0, 1200.0, 350.0)

Sur → List(45.0, 1200.0, 350.0)
Norte → List(1200.0, 350.0, 25.0, 45.0)


gruposPorRegion: org.apache.spark.rdd.RDD[(String, Iterable[Double])] = ShuffledRDD[13] at groupByKey at cmd8.sc:1

In [9]:
import org.apache.spark.HashPartitioner

// Reparticionar en 2 particiones con hash de la clave
val ventasParticionadas = ventasPorRegion.partitionBy(new HashPartitioner(2))

println(s"Número de particiones: ${ventasParticionadas.getNumPartitions}")
// Número de particiones: 2

// Ver qué hay en cada partición
ventasParticionadas.mapPartitionsWithIndex { (idx, iter) =>
  iter.map(par => s"Partición $idx: $par")
}.collect().foreach(println)

Número de particiones: 2
Partición 0: (Norte,1200.0)
Partición 0: (Sur,45.0)
Partición 0: (Norte,350.0)
Partición 0: (Sur,1200.0)
Partición 0: (Norte,25.0)
Partición 0: (Sur,350.0)
Partición 0: (Norte,45.0)


import org.apache.spark.HashPartitioner
ventasParticionadas: org.apache.spark.rdd.RDD[(String, Double)] = ShuffledRDD[14] at partitionBy at cmd9.sc:4

In [10]:
import org.apache.spark.RangePartitioner

// RDD con clave numérica: (año, ventas)
val ventasAnuales = sc.parallelize(List(
  (2020, 15000.0), (2021, 18500.0), (2022, 22000.0),
  (2023, 19000.0), (2024, 25000.0), (2019, 12000.0)
))

val rangePartitioner = new RangePartitioner(3, ventasAnuales)
val ventasRango = ventasAnuales.partitionBy(rangePartitioner)

ventasRango.mapPartitionsWithIndex { (idx, iter) =>
  iter.map(par => s"Partición $idx: $par")
}.collect().sorted.foreach(println)
// Partición 0: (2019,12000.0)
// Partición 0: (2020,15000.0)
// Partición 1: (2021,18500.0)
// Partición 1: (2022,22000.0)
// Partición 2: (2023,19000.0)
// Partición 2: (2024,25000.0)

Partición 0: (2019,12000.0)
Partición 0: (2020,15000.0)
Partición 1: (2021,18500.0)
Partición 1: (2022,22000.0)
Partición 2: (2023,19000.0)
Partición 2: (2024,25000.0)


import org.apache.spark.RangePartitioner
ventasAnuales: org.apache.spark.rdd.RDD[(Int, Double)] = ParallelCollectionRDD[16] at parallelize at cmd10.sc:4
rangePartitioner: RangePartitioner[Int, Double] = org.apache.spark.RangePartitioner@1f0cec
ventasRango: org.apache.spark.rdd.RDD[(Int, Double)] = ShuffledRDD[19] at partitionBy at cmd10.sc:10

In [11]:
// Dataset de ventas: (producto, región, importe, unidades)
val datosVentas = sc.parallelize(List(
  ("Laptop",    "Norte",  1200.0, 3),
  ("Teclado",   "Sur",      45.0, 10),
  ("Monitor",   "Norte",   350.0, 5),
  ("Laptop",    "Sur",    1200.0, 2),
  ("Ratón",     "Norte",    25.0, 20),
  ("Monitor",   "Centro",  350.0, 4),
  ("Teclado",   "Norte",    45.0, 8),
  ("Laptop",    "Centro", 1200.0, 1),
  ("Auriculares","Sur",     80.0, 6),
  ("Ratón",     "Centro",   25.0, 15),
  ("Auriculares","Norte",   80.0, 9),
  ("Teclado",   "Centro",   45.0, 12)
))

println(s"Total registros: ${datosVentas.count()}")

2026-04-27T09:10:02.472589500Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

datosVentas: org.apache.spark.rdd.RDD[(String, String, Double, Int)] = ParallelCollectionRDD[21] at parallelize at cmd11.sc:2

In [12]:
// Paso 1: crear Pair RDD (región, importe_total_línea)
// El importe de cada línea es: importe_unitario * unidades
val importesPorRegion = datosVentas.map { case (producto, region, precio, uds) =>
  (region, precio * uds)
}

// Paso 2: sumar importes por región
val totalPorRegion = importesPorRegion.reduceByKey(_ + _)

println("=== Total de ventas por región ===")
totalPorRegion.sortBy(_._2, ascending = false).collect().foreach {
  case (region, total) => println(f"  $region%-10s → $total%,.2f €")
}

=== Total de ventas por región ===
  Norte      → 6.930,00 €
  Centro     → 3.515,00 €
  Sur        → 3.330,00 €


importesPorRegion: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[22] at map at cmd12.sc:3
totalPorRegion: org.apache.spark.rdd.RDD[(String, Double)] = ShuffledRDD[23] at reduceByKey at cmd12.sc:8

In [13]:
// Usamos aggregateByKey para calcular (suma, count) y luego la media
val mediasPorRegion = importesPorRegion.aggregateByKey(
  (0.0, 0)
)(
  (acc, v) => (acc._1 + v, acc._2 + 1),
  (a, b)   => (a._1 + b._1, a._2 + b._2)
)

println("=== Media de importe por venta y región ===")
mediasPorRegion.collect().foreach { case (region, (suma, count)) =>
  val media = suma / count
  println(f"  $region%-10s → media $media%,.2f € (sobre $count transacciones)")
}

=== Media de importe por venta y región ===
  Sur        → media 1.110,00 € (sobre 3 transacciones)
  Norte      → media 1.386,00 € (sobre 5 transacciones)
  Centro     → media 878,75 € (sobre 4 transacciones)


mediasPorRegion: org.apache.spark.rdd.RDD[(String, (Double, Int))] = ShuffledRDD[29] at aggregateByKey at cmd13.sc:4

In [14]:
// Usamos combineByKey para obtener la lista de productos por región
val productosPorRegion = datosVentas.map { case (producto, region, _, _) =>
  (region, producto)
}.combineByKey(
  (v: String) => List(v),
  (acc: List[String], v: String) => (acc :+ v).distinct,
  (acc1: List[String], acc2: List[String]) => (acc1 ++ acc2).distinct
)

println("=== Catálogo de productos por región ===")
productosPorRegion.collect().foreach { case (region, productos) =>
  println(s"  $region → ${productos.sorted.mkString(", ")}")
}

=== Catálogo de productos por región ===
  Sur → Auriculares, Laptop, Teclado
  Norte → Auriculares, Laptop, Monitor, Ratón, Teclado
  Centro → Laptop, Monitor, Ratón, Teclado


productosPorRegion: org.apache.spark.rdd.RDD[(String, List[String])] = ShuffledRDD[31] at combineByKey at cmd14.sc:7

🔹 Ejercicio 2 — Medias por categoría con combineByKey


In [15]:
// (módulo, nota)
val notasEstudiantes = sc.parallelize(List(
  ("Scala",  8.5), ("Spark",  7.0), ("Scala",  9.0),
  ("BigData",6.5), ("Spark",  8.5), ("Scala",  7.5),
  ("BigData",9.0), ("Spark",  6.0), ("Scala",  8.0),
  ("BigData",7.5), ("Spark",  9.5), ("BigData",8.0),
  ("Scala",  9.5), ("Spark",  7.5), ("BigData",5.5)
))

println(s"Total evaluaciones: ${notasEstudiantes.count()}")

Total evaluaciones: 15


notasEstudiantes: org.apache.spark.rdd.RDD[(String, Double)] = ParallelCollectionRDD[32] at parallelize at cmd15.sc:2

In [16]:
// Acumulador: (suma, máximo, count)
// Tipo de combinación: (Double, Double, Int)

val estadisticasPorModulo = notasEstudiantes.combineByKey(
  // createCombiner: primer valor → (suma, max, count=1)
  (nota: Double) => (nota, nota, 1),

  // mergeValue: actualizar acumulador con un nuevo valor
  (acc: (Double, Double, Int), nota: Double) =>
    (acc._1 + nota, scala.math.max(acc._2, nota), acc._3 + 1),

  // mergeCombiners: unir dos acumuladores de distintas particiones
  (a: (Double, Double, Int), b: (Double, Double, Int)) =>
    (a._1 + b._1, scala.math.max(a._2, b._2), a._3 + b._3)
)

println("=== Estadísticas por módulo ===")
println(f"${"Módulo"}%-12s ${"Media"}%8s ${"Máximo"}%8s ${"Evaluc."}%8s")
println("-" * 42)

estadisticasPorModulo.collect().sortBy(_._1).foreach {
  case (modulo, (suma, maximo, count)) =>
    val media = suma / count
    println(f"$modulo%-12s $media%8.2f $maximo%8.1f $count%8d")
}

=== Estadísticas por módulo ===
Módulo          Media   Máximo  Evaluc.
------------------------------------------
2026-04-27T09:18:14.859048500Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:

estadisticasPorModulo: org.apache.spark.rdd.RDD[(String, (Double, Double, Int))] = ShuffledRDD[33] at combineByKey at cmd16.sc:13

🔹 Ejercicio 3 — Contar clics por usuario en un log simulado

In [17]:
// Log de eventos: "usuario,sección,segundos"
val logEventos = sc.parallelize(List(
  "u001,inicio,12",    "u002,cursos,45",   "u001,cursos,120",
  "u003,inicio,8",     "u001,video,300",   "u002,video,250",
  "u003,cursos,90",    "u004,inicio,5",    "u002,inicio,15",
  "u001,foro,60",      "u003,video,180",   "u004,cursos,75",
  "u002,foro,40",      "u003,foro,30",     "u004,video,200",
  "u001,cursos,95",    "u004,foro,55",     "u003,inicio,10",
  "u002,cursos,110",   "u001,inicio,20"
))

println(s"Total eventos en el log: ${logEventos.count()}")

Total eventos en el log: 20


logEventos: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[34] at parallelize at cmd17.sc:2

In [18]:
// Paso 1: convertir a Pair RDD (usuario, segundos)
val clicsPorUsuario = logEventos.map { linea =>
  val campos  = linea.split(",")
  val usuario = campos(0)
  val segundos = campos(2).toInt
  (usuario, segundos)
}

// Paso 2: aggregateByKey para (total_segundos, num_clics, max_sesion)
val metricasPorUsuario = clicsPorUsuario.aggregateByKey(
  (0, 0, 0)   // (total_seg, clics, max_seg)
)(
  (acc, seg) => (acc._1 + seg, acc._2 + 1, scala.math.max(acc._3, seg)),
  (a, b)     => (a._1 + b._1, a._2 + b._2, scala.math.max(a._3, b._3))
)

println("=== Métricas de actividad por usuario ===")
println(f"${"Usuario"}%-10s ${"Clics"}%7s ${"Total(s)"}%10s ${"Max(s)"}%8s ${"Media(s)"}%10s")
println("-" * 50)

metricasPorUsuario.collect().sortBy(_._1).foreach {
  case (usuario, (total, clics, maxSeg)) =>
    val media = total.toDouble / clics
    println(f"$usuario%-10s $clics%7d $total%10d $maxSeg%8d $media%10.1f")
}

=== Métricas de actividad por usuario ===
Usuario      Clics   Total(s)   Max(s)   Media(s)
--------------------------------------------------
u001             6        607      300      101,2
u002             5        460      250       92,0
u003             5        318      180       63,6
u004             4        335      200       83,8


clicsPorUsuario: org.apache.spark.rdd.RDD[(String, Int)] = MapPartitionsRDD[35] at map at cmd18.sc:2
metricasPorUsuario: org.apache.spark.rdd.RDD[(String, (Int, Int, Int))] = ShuffledRDD[36] at aggregateByKey at cmd18.sc:12

In [19]:
// Crear Pair RDD (usuario, total_segundos) y ordenar
val rankingUsuarios = metricasPorUsuario.mapValues { case (total, _, _) => total }

println("=== Ranking de usuarios por tiempo en plataforma ===")
rankingUsuarios
  .sortBy(_._2, ascending = false)
  .collect()
  .zipWithIndex
  .foreach { case ((usuario, total), idx) =>
    val minutos = total / 60
    val segs    = total % 60
    println(f"  ${idx + 1}. $usuario%-8s → ${total}s (${minutos}m ${segs}s)")
  }

=== Ranking de usuarios por tiempo en plataforma ===
  1. u001     → 607s (10m 7s)
  2. u002     → 460s (7m 40s)
  3. u004     → 335s (5m 35s)
  4. u003     → 318s (5m 18s)


rankingUsuarios: org.apache.spark.rdd.RDD[(String, Int)] = MapPartitionsRDD[37] at mapValues at cmd19.sc:2

In [20]:
// Objetivo: obtener todos los pares (usuario, sección) del log
val seccionesPorUsuario = logEventos.map { linea =>
  val campos = linea.split(",")
  (campos(0), campos(1))   // (usuario, sección)
}

// Secciones únicas visitadas por cada usuario
val seccionesUnicas = seccionesPorUsuario
  .groupByKey()
  .mapValues(secciones => secciones.toSet.toList.sorted)

println("=== Secciones visitadas por usuario ===")
seccionesUnicas.collect().sortBy(_._1).foreach { case (usuario, secciones) =>
  println(s"  $usuario → ${secciones.mkString(", ")}")
}

=== Secciones visitadas por usuario ===
2026-04-27T09:25:06.572717200Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig

seccionesPorUsuario: org.apache.spark.rdd.RDD[(String, String)] = MapPartitionsRDD[43] at map at cmd20.sc:2
seccionesUnicas: org.apache.spark.rdd.RDD[(String, List[String])] = MapPartitionsRDD[45] at mapValues at cmd20.sc:10

🔹 Ejercicio 4 — mapValues vs map con particionado

In [ ]:
// Demostración: mapValues preserva el particionador, map no
val rddParticionado = ventasPorRegion
  .partitionBy(new HashPartitioner(4))

println(s"Particionador original: ${rddParticionado.partitioner}")
// Some(org.apache.spark.HashPartitioner@4)

val conMapValues = rddParticionado.mapValues(_ * 1.1)
println(s"Particionador tras mapValues: ${conMapValues.partitioner}")
// Some(org.apache.spark.HashPartitioner@4)   ← ✅ Se preserva

val conMap = rddParticionado.map { case (k, v) => (k, v * 1.1) }
println(s"Particionador tras map: ${conMap.partitioner}")
// None   ← ⚠️ Se pierde el particionado

println("""
  Conclusión:
  - mapValues preserva el particionador → menos shuffles en operaciones posteriores
  - map destruye el particionador → puede causar shuffles innecesarios
""")